In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve, schur
import scipy as sp
from scipy.integrate import solve_ivp
import sys, os, pickle
from joblib import Parallel, delayed, cpu_count
from matplotlib.animation import FuncAnimation
from scripts.utility import orbit, call_method
from scripts.models import Kuramoto
import datetime

In [ ]:
if __name__ == "__main__":
    param_file = "./config_models/kuramoto_param.in"  # JSON file containing model parameters
    model = Kuramoto(param_file)
    # print("Loaded parameters:", model.n_z)

    model.n_z = 100
    model.xmin = -np.pi
    model.xmax = np.pi
    # model.kuramoto_potential = lambda z : -np.cos(z - model.alpha_shift) + .5*np.cos(2*(z -model.alpha_shift))
    # model.kuramoto_potential = lambda z : -np.cos(z) + 0.5*np.cos(2*(z -model.alpha_shift))
    # model.alpha_shift = np.pi/3
    # # model.kuramoto_potential = lambda z : -np.cos(z) + 0.5*np.cos(2*(z -model.alpha_shift))
    model.update_params()

    # f = model.dydt
    # J = model.jacobian

    z, z_centers, h = model.mesh1D  # Get the mesh and centers


In [ ]:
model.alpha_shift = np.pi/3
model.nu=0.2
y0 = np.full(len(z_centers), 1/(2*np.pi)) + 1e-1*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
# plt.plot(z_centers, model.V_kuramoto(rho=y0, z=z_centers-z_centers[0]))
# plt.show()
plt.plot(z_centers, model.kuramoto_potential(z_centers))
# plt.plot(z_centers, -np.cos(z_centers) + 0.2*np.cos(2*(z_centers - np.pi/3)), color="red")

In [ ]:
denom = 3
alpha = np.pi/denom
Ic = 2/np.cos(alpha)
y0 = (1/(2*np.pi))*np.ones_like(z_centers)+0.001*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
model.alpha_shift = alpha
I_values = [0.9*Ic, 1.1*Ic, 2*Ic] #[0.5*Ic, 0.9*Ic, 1.1*Ic, 2*Ic]
coeff_labels = ["0.9*I_c","1.1*I_c","2*I_c"]#["0.5*I_c", "0.9*I_c", "1.1*I_c", "2*I_c", "5*I_c"]
fig, axes = plt.subplots(1, len(I_values), figsize=(14, 4), sharex=True, sharey=True)
# fig,ax = plt.subplots()
sols = []
lines = []

for ax, I, coeff in zip(axes, I_values, coeff_labels):
# I=2*Ic
    model.I = I
# coeff="2*Ic"
    sol = solve_ivp(model.dydt, (0, 20*model.T_ini), y0, method='BDF', jac = model.jacobian,
                    rtol=1e-7, atol=1e-9,
                    t_eval=np.linspace(0, 20*model.T_ini, 1000))
    sols.append(sol)
    print(f"Computed solution for I={I:.2f}")
    line, = ax.plot(z_centers, sol.y[:, 0], label=rf"$\alpha = \dfrac{{\pi}}{{{denom}}},\ I={coeff}$") #\dfrac{{\pi}}{{{denom}}}
    lines.append(line)
    ax.set_xlim(z_centers[0], z_centers[-1])
    ax.set_ylim(0.8 * np.min(sol.y), np.max(sol.y) * 1.1)
    ax.set_xlabel('z')
    ax.set_ylabel(r'$\rho(z)$')
    ax.set_title(rf'$I={coeff}$')
    ax.legend(loc='upper right')

fig.suptitle(rf'Evolution of $\rho(z,t)$ for different values of $I$ with $\alpha = \dfrac{{\pi}}{{{denom}}}, I_c = 2 \times sec(\alpha)$', fontsize=16)
fig.tight_layout()


def update(frame):
    for ax, line, sol, coeff in zip(axes, lines, sols, coeff_labels):
        line.set_ydata(sol.y[:, frame])
        ax.set_title(rf'$I={coeff},\ t={sol.t[frame]:.2f}$')

    return lines

ani = FuncAnimation(fig, update, frames=len(sols[0].t), blit=True, interval=50) 
ani.save(f'sol_with_two_mode_kuram_alpha_{alpha:.2f}_bdf.gif', writer='pillow')
plt.show()
   


In [ ]:

for coeff, sol in zip(coeff_labels, sols):
    first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
    plt.plot(sol.t, first_moment, label=rf"$I={coeff}$")
    plt.xlabel('t', fontsize=18)
    plt.ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
    plt.xlim(10,14)
    plt.legend()
    plt.show()
    plt.plot(sol.y[0, :], sol.y[10, :],marker='.',linestyle='None', label=rf"$I={coeff}$")
    plt.xlabel(r'$y_j$', fontsize=18)
    plt.ylabel(r'$y_i$', fontsize=14)
    plt.legend()
    plt.show()

In [ ]:
Denom = [3, 4, 5, 6]
sol_wrt_alpha = []
periods = []
for denom in Denom:
    alpha = np.pi/denom
    Ic = 2/np.cos(alpha)
    y0 = (1/(2*np.pi))*np.ones_like(z_centers)+0.0001*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
    # y0 = sol.y[:,-1]
    model.alpha_shift = alpha
    model.I = 4 * Ic
    coeff = "4*I_c"
    Tc = (2*np.pi)/np.tan(alpha)
    periods.append(Tc)
    sol = solve_ivp(f, (0, 10*model.T_ini), y0, method='BDF', jac = J,
                rtol=1e-7, atol=1e-9,
                t_eval=np.linspace(0, 10*model.T_ini, 3000))
    sol_wrt_alpha.append(sol)
    fig = plt.figure(figsize=(6, 4))
    # line, = plt.plot(z_centers, sol.y[:, 0])
    # plt.xlim(z_centers[0], z_centers[-1])
    # plt.ylim(0.8 * np.min(sol.y), np.max(sol.y) * 1.1)
    # plt.xlabel('z')
    # plt.ylabel(r'$\rho(z)$')
    # plt.title(rf'$I={coeff}$')
    # # plt.legend(loc='upper right')

    # fig.suptitle(rf'Evolution of $\rho(z,t)$ for different values of $I$ with $\alpha = \dfrac{{\pi}}{{{denom}}}, I_c = 2 \times sec(\alpha)$', fontsize=16)
    # fig.tight_layout()


    # def update(frame):
    #     line.set_ydata(sol.y[:, frame])
    #     plt.title(rf'$I={coeff},\ t={sol.t[frame]:.2f}$')

    #     return line,

    # ani = FuncAnimation(fig, update, frames=len(sol.t), blit=True, interval=50) 
    # ani.save(f'solution_with_kuramoto_potential_alpha_{alpha:.2f}_{coeff}_bdf.gif', writer='pillow')
    # plt.show()

    first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
    plt.plot(sol.t, first_moment, color='green', linewidth=1.5, label=r'$\int_{\Omega} z \rho(z,t)dz $')
# plt.plot(sol.t[0:], np.mean(sol.y, axis=0)[0:], color='blue', linewidth=1.5)
    plt.xlabel('t', fontsize=18)
    plt.ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
# ax.set_ylabel(r'$\max_z ( \rho(z,t) )$', fontsize=14)
    plt.title(rf'$I={coeff}$')
# plt.xlim(60, 62)

    # ax.set_xlim([15,20])
# plt.ylim([10, np.max(sol.y)*1.1])
    plt.legend()
    plt.savefig(f'first_moment_kuramoto_alpha_{alpha:.2f}_{coeff}_bdf.png', dpi=300)
    plt.show()


    

In [ ]:

first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
plt.plot(sol.t, first_moment, color='green', linewidth=1.5, label=r'$\int_{\Omega} z \rho(z,t)dz $')
# plt.xlim(10.1,10.)
# plt.plot

In [ ]:
# periods = [0.6,1.0,1.3,1.6]
periods

In [ ]:
def run(model,n_z,orbit_method,p0,T, y0):
    epsilon = model.precision
    model.n_z = n_z
    model.p0 = p0 #Size of the dominant subspace
    model.update_params()#(**{'n_z': n_z}) #Update the model parameters
    #Initialization
    _, _, h = model.mesh1D
    
    T_unit = 1.0
    t_span = (0, 4*T)
    H = h*np.ones_like(y0)
    model.m0 = H @ y0
    print('Mass at initial point:', model.m0)
    #We integrate sufficiently the equation to find a good starting point
    phi_t = solve_ivp(model.dydt, t_span, y0, method='BDF', jac = model.jacobian,
                     rtol=1e-7, atol=1e-9,
                     t_eval= [4*T])#np.linspace(0, 10, 100))
    
    y_T = phi_t.y[:,-1] #Using phi(y0,T0) as a starting point
    
  
    print('Mass at the starting point:', H@y_T)
    orbit_finder = orbit(model.dydt,y0,T, model.jacobian, solve_ivp, model.method, 10000,model.max_iter, epsilon)
    
    V_0 = np.eye(len(y0))[:,:p0+model.pe]#Initial guess of the subspace
    #The arguments to pass to the orbit_finder method
    args_func = {
    "y_0": y_T,
    "T_0": T,
    "model": model,
    "f_unscaled": model.dydt,
    "jac_unscaled": model.jacobian,
    "alpha_0": model.alpha,
    "Max_iter": model.max_iter,
    "epsilon": epsilon,
    "subsp_iter": model.subsp_iter,
    "l": model.picard_iter,
    "Ve_0": V_0,
    "p0": p0,
    "pe": model.pe,
    "rho": model.rho,
    "phase_cond": 2,
    "l": model.picard_iter,
    "full_sub_iter": 0, #model.full_sub_iter, # Use the full subspace iteration if True for the subspace iteration with projection
    "h": h
    }
    method_to_call= getattr(orbit_finder, orbit_method)

    return call_method(method_to_call, **args_func)

In [ ]:
with open('./config_models/init_two_mod_kuramoto_2026-06-12_alpha_pi_over_3.pkl', 'rb') as fic:
    data = pickle.load(fic)

data['T']
data['coeff']

In [ ]:
today = datetime.date.today().strftime("%Y-%m-%d")
# for sol, denom, period in zip(sol_wrt_alpha, Denom, periods):
denom = 3
# sol = sol_wrt_alpha[0]
alpha = np.pi/denom
coeff = "2*I_c"
Ic = 2/np.cos(alpha)
model.I = 2.0 * Ic
model.alpha_shift = np.pi/3

model.update_params()
y0 = (1/(2*np.pi))*np.ones_like(z_centers)+ 0.001*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))

soly = solve_ivp(model.dydt, (0, 2*model.T_ini), y0, method='BDF', jac = model.jacobian,
                    rtol=1e-7, atol=1e-9,
                    t_eval=[2*model.T_ini])
y0 = soly.y[:, -1]
# y0 = (1/(2*np.pi))*np.ones_like(z_centers)+0.1*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
T = 3.6
model.precision = 1e-10
print(rf"Running orbit finding for $\alpha = \dfrac{{\pi}}{{{denom}}}$")
k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged, mass, monodromy = run(model = model, n_z=model.n_z, 
                                                                                        orbit_method='Newton_mass_conserv4', p0= model.p0, T=T, y0=y0)


# #save the results to a pickle file
# with open(f'./Results/kuramoto_param.in/NP_converg_{today}.pkl', 'wb') as file:
#      
#     pickle.dump(data, file)
# with open(f'./config_models/init_two_mod_kuramoto_{today}_alpha_pi_over_{denom}.pkl', 'wb') as file:
#     data = {
#         'y_0': y_by_iter[k],
#         'T': T_by_iter[k],
#         'alpha_shift': model.alpha_shift,
#         'alpha_shift_2': model.alpha_shift_2,
#         'I': model.I,
#         'coeff': coeff
#     }
#     pickle.dump(data, file)

In [ ]:
sol = solve_ivp(f, (0, 20*T_by_iter[k]), y_by_iter[k], method='BDF', jac = J,
                    rtol=1e-7, atol=1e-9,
                    t_eval=np.linspace(0, 20*T_by_iter[k], 1000))
first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
plt.plot(sol.t, first_moment, label=rf"$I={coeff}$")
plt.xlabel('t', fontsize=18)
plt.ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
# plt.xlim(30,35)
plt.legend()
plt.show()

print('max rho at the end of the orbit:', np.max(sol.y[:,-1]))
sol_pred = sols[-1]
print('max sols[-1]', np.max(sol_pred.y[:,-1]))
np.max(sol.y[:,-1]) - np.max(sol_pred.y[:,-1])

In [ ]:
#Load the results from the pickle file
with open(f'./Results/kuramoto_param.in/Newton_converg_{today}.pkl', 'rb') as file:
    data = pickle.load(file)
    k = data['k']
    T_by_iter = data['T_by_iter']
    y_by_iter = data['y_by_iter']
    Norm_B = data['Norm_B']
    Abs_Err = data['Abs_Err']
    Rel_Err = data['Rel_Err']
    converged = data['converged']                
    mass = data['Delta_mass']
    p0 = data['p0']
    alpha = data['alpha']
    n_z = data['n_z']

with open(f'./Results/kuramoto_param.in/NP_converg_{today}.pkl', 'rb') as file:
    data2 = pickle.load(file)
    k2 = data2['k']
    T_by_iter2 = data2['T_by_iter']
    y_by_iter2 = data2['y_by_iter']
    Norm_B2 = data2['Norm_B']
    Abs_Err2 = data2['Abs_Err']
    Rel_Err2 = data2['Rel_Err']
    converged2 = data2['converged']                
    mass2 = data2['Delta_mass']
    p02 = data2['p0']
    alpha2 = data2['alpha']
    n_z2 = data2['n_z'] 
fig = plt.figure(figsize=(22/2.54, 28/2.54), dpi=200)


plt.rcParams.update({'font.size': 18})
ax1 = fig.add_subplot(211)
# ax1.loglog(range(k+1), Norm_B[:k+1],'x--',color='blue', label='Residual Norm')
# ax1.loglog(range(k+1), Abs_Err[:k+1],'x--',color='red', label=r'$||y_k - y_{k-1}||_{\infty}$')
ax1.loglog(range(k+1), Rel_Err[:k+1], 'x--',color='green',label=r'Newton')
ax1.loglog(range(k2+1), Rel_Err2[:k2+1], 'x--',color='red',label=r'NP')

# ax1.loglog(np.arange(k+1), 2.**(-2.**np.arange(k+1)), '*-', label=r'$2^{-2^k}$')
ax1.set_xlabel('Iteration')
ax1.set_ylabel(r'$\dfrac{||y_k - y_{k-1}||_{\infty}}{||y_k||_{\infty}}$', fontsize=20)
# fig.suptitle(f"Convergence check of the NP method and evolution of mass.")
ax1.set_title(r'Convergence of the Newton and NP methods') # $\dot{y} = f(y) + \alpha \nabla_y m(y)$')
# ax1.set_xticks(range(0, k+1, 2))  # Adjust '2' to your desired step size
# ax1.set_yticks([1e-13,1e-9,1e-3], fontsize= 18)
# plt.tick_params(axis='both', labelsize=20)
ax1.tick_params(axis='both')
# ax1.grid(visible=True, which='both', linestyle='--', linewidth=0.5)
ax1.legend(loc='best')
# ax2 = fig.add_subplot(212)
# ax2.plot(range(k+1), mass[:k+1], marker='o', label = 'Newton')
# ax2.plot(range(k2+1), mass2[:k2+1], marker='+', label='NP')
# ax2.legend()
# ax2.set_xlabel('Iteration', fontsize=20)
# ax2.set_ylabel(r'$m(y) = y \cdot v$', fontsize = 20)
# ax2.set_title(f'$m(y)$ evolution over Iterations. Grid size $n_z$={model.n_z}')
# ax2.tick_params(axis='both', labelsize=20)

# # ax2.set_xticks(range(0, k2+1, 2))  # Adjust '2' to your desired step size
# ax2.grid(visible=True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig(f'./Results/kuramoto_param.in/newton_cvrg_nz_{model.n_z}.png')
plt.show()  

In [ ]:
first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
plt.plot(sol.t, first_moment, color='green', linewidth=1.5, label=r'$\int_{\Omega} z \rho(z,t)dz $')
# plt.plot(sol.t[0:], np.mean(sol.y, axis=0)[0:], color='blue', linewidth=1.5)
plt.xlabel('t', fontsize=18)
plt.ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
# ax.set_ylabel(r'$\max_z ( \rho(z,t) )$', fontsize=14)
plt.title(rf'$I={coeff}$')
# plt.xlim(60, 62)

    # ax.set_xlim([15,20])
# plt.ylim([10, np.max(sol.y)*1.1])
plt.legend()
# plt.suptitle(r'Periodicity evolution of the density with $\alpha = {alpha},\quad I_c = 2 \times sec(\alpha)$', fontsize=16)
plt.tight_layout()

fig = plt.figure(figsize=(12/2.54, 8/2.54), dpi=200)
plt.imshow(
    sol.y,
    aspect='auto',
    cmap='Blues',
    origin='lower',
    extent=(
        float(np.min(sol.t)),
        float(np.max(sol.t)),
        float(np.min(z_centers)),
        float(np.max(z_centers)),
    )
)

plt.colorbar(label=r'$\rho(t,z)$')
plt.xlabel("t", fontsize = 14)
plt.ylabel('z', fontsize=14)
# plt.ylim(-10,7)
# plt.xlim(10, 18)
# ax[0].set_title(f'Density $\rho(t,z) at I = {I_vals[ind]:.4f}')
plt.tight_layout()
# plt.savefig(rf'surface_map_density_alpha_{alpha}_{coeff}.png')
plt.show()
# Phase space plot
plt.figure(figsize=(6, 4))
plt.plot(sol.y[0,0:], sol.y[17,0:])

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 4), sharex=True, sharey=True)

for sol, I, coeff, ax in zip(sols, I_values, coeff_labels, axes):
    first_moment = np.sum(sol.y * z_centers[:, None], axis=0) * h
    ax.plot(sol.t, first_moment, color='green', linewidth=1.5, label=r'$\int_{\Omega} z \rho(z,t)dz $')
    # ax.plot(sol.t[0:], np.max(sol.y, axis=0)[0:], color='blue', linewidth=1.5)
    ax.set_xlabel('t', fontsize=18)
    ax.set_ylabel(r'$\int_{\Omega} z \rho(z,t)dz $', fontsize=14)
    # ax.set_ylabel(r'$\max_z ( \rho(z,t) )$', fontsize=14)
    ax.set_title(rf'$I={coeff}$')

    # ax.set_xlim([15,20])
# plt.ylim([10, np.max(sol.y)*1.1])
    ax.legend()
plt.suptitle(rf'First moment of the density with $\alpha =  \dfrac{{\pi}}{{{denom}}},\quad I_c = 2 \times sec(\alpha)$', fontsize=16)
plt.tight_layout()
plt.savefig(f'solution_first_moment_with_kuramoto_potential_alpha_{alpha:.2f}_bdf.png')
plt.show()

In [ ]:
nz = model.n_z
model.alpha_shift = np.pi/3
Ic = 2/np.cos(model.alpha_shift)
model.I = 1.1*Ic
model.alpha = 0.0 #Articial parameter for conservativity
# model.p0 = 10
# sol = sols[-2]
y0 = sol.y[:,-1]#+0.005*np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))
T = 3.1
model.precision = 1e-6
k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged, mass, monodromy = run(model,f,J,nz,"NP_mass_conserv_scal",
                                                                         model.p0,T,y0, filename=None, t_scale=True);

#save the results to a pickle file
# with open('./Results/mckean_vlasov_param_1.in/Np_spence_18_03_2026.pkl', 'wb') as file:
#     data = {
#         'k': k,
#         'T_by_iter': T_by_iter,
#         'y_by_iter': y_by_iter,
#         'Norm_B': Norm_B,
#         'Abs_Err': Abs_Err,
#         'Rel_Err': Rel_Err,
#         'converged': converged,                
#         'Delta_mass': mass,
#         'p0': model.p0,
#         'alpha': model.alpha,
#         'n_z': model.n_z
#     }
#     pickle.dump(data, file)

In [ ]:
T = T_by_iter[k]
y0 = y_by_iter[k]
sol = solve_ivp(f, (0, 10*T), y0, method='BDF', jac = J,
                rtol=1e-7, atol=1e-9,
                t_eval=np.linspace(0, 10*T, 1000))


fig = plt.figure(figsize=(6, 4))
line, = plt.plot(z_centers, sol.y[:, 0], label=rf"$\alpha = \dfrac{{\pi}}{{{denom}}},\ I={coeff}$")
plt.xlim(z_centers[0], z_centers[-1])
plt.ylim(0.8 * np.min(sol.y), np.max(sol.y) * 1.1)
plt.xlabel('z')
plt.ylabel(r'$\rho(z)$')
plt.title(rf'$I={coeff}$')
plt.legend(loc='upper right')

fig.suptitle(rf'Evolution of $\rho(z,t)$ for different values of $I$ with $\alpha = \dfrac{{\pi}}{{{denom}}}, I_c = 2 \times sec(\alpha)$', fontsize=16)
fig.tight_layout()


def update(frame):
    line.set_ydata(sol.y[:, frame])
    plt.title(rf'$I={coeff},\ t={sol.t[frame]:.2f}$')

    return line,

ani = FuncAnimation(fig, update, frames=len(sol.t), blit=True, interval=50) 
ani.save(f'solution_with_kuramoto_potential_alpha_{alpha:.2f}_{coeff}_bdf.gif', writer='imagemagick')
plt.show()
fig = plt.figure(figsize=(12/2.54, 8/2.54), dpi=200)
plt.imshow(
    sol.y,
    aspect='auto',
    cmap='Blues',
    origin='lower',
    extent=(
        float(np.min(sol.t)),
        float(np.max(sol.t)),
        float(np.min(z_centers)),
        float(np.max(z_centers)),
    )
)

plt.colorbar(label=r'$\rho(t,z)$')
plt.xlabel("t", fontsize = 14)
plt.ylabel('z', fontsize=14)
# plt.ylim(-10,7)
# plt.xlim(10, 18)
# ax[0].set_title(f'Density $\rho(t,z) at I = {I_vals[ind]:.4f}')
plt.tight_layout()
# plt.savefig(rf'surface_map_density_alpha_{alpha}_{coeff}.png')
plt.show()

In [ ]:
today = datetime.date.today().strftime("%Y_%m_%d")
with open(f'./config_models/init_kuramoto_{today}.pkl', 'wb') as file:
    data = {
        'y_0': y0,
        'T':T,
        'alpha_shift': model.alpha_shift,
        'I': model.I,
        'coeff': coeff
    }
    pickle.dump(data, file)

In [ ]:
with open(f'./config_models/init_kuramoto_2026-05-22_alpha_pi_over_6.pkl', 'rb') as file:
    data = pickle.load(file)
    y0_loaded = data['y_0']
    T_loaded = data['T']
    alpha_shift_loaded = data['alpha_shift']
    I_loaded = data['I']
    coeff_loaded = data['coeff']

In [ ]:
coeff_loaded

In [ ]:
#Read a pickle file
import pickle
with open('./Results/kuramoto_param.in/branch_solutions_2026-05-21.pkl', 'rb') as fic:
    solutions= pickle.load(fic)

In [ ]:
#Plot the bifurcation diagram
# model.I,model.alpha, y_0, T, mass[k],Rel_Err[k],k
I_vals = [sol[0] for sol in solutions]
T_sols = [sol[3] for sol in solutions]
y_sol = [sol[2] for sol in solutions]
precision = [sol[5] for sol in solutions]
Mass = [sol[4] for sol in solutions]
Monodromy = [sol[7] for sol in solutions]
# Precision = [sol[5][k] for sol, k in zip(solutions, k_iter)]
# mass = [Mass[j][k] for k, j in zip(k_iter, range(len(k_iter)))]
y_sol_max = [np.max(sol[2]) for sol in solutions]
# y_sol_norm = [np.linalg.norm(sol[1], ord=2) for sol in solutions]
# y_sol_mass = [h*np.ones_like(sol[1]) @ sol[1] for sol in solutions]
# plt.figure(figsize=(18/2.54,18/2.54))
fig, ax = plt.subplots(1, 2, figsize=(38/2.54,16/2.54), dpi=300)
# plt.suptitle('Evolution of the periode T wrt to bifurcation parameter I', fontsize=16)
ax[0].plot(I_vals, T_sols, color='blue', marker='o', linestyle='-', linewidth=1.5, markersize=3, label='Period T vs Intensity I', zorder=2)
ax[0].set_title('Bifurcation Diagram: Period T vs Intensity I')
ax[0].set_xlabel('I', fontsize = 14)
ax[0].set_ylabel('T', fontsize = 14)
ax[0].tick_params(axis='both', labelsize=16)
# ax[0].grid()
# ax[0].legend()
ax[1].plot(I_vals, precision,  color='orange', marker='o', linestyle='--', linewidth=2, markersize=3, label=f'$m(y)=H \cdot y(0)$ vs Intensity I')
ax[1].set_title(f'Bifurcation Diagram: Mass $m(y)=v \cdot y(0)$ vs Intensity I')
# ax[1].set_yticks((np.arange(0.0,.2,0.05)*1e-8))
ax[1].set_xlabel('I', fontsize = 14)
ax[1].set_ylabel(f'$m(y)$', fontsize=14)
# ax[1].grid()
plt.tight_layout()
# plt.legend()
# plt.savefig('./Results/mckean_vlasov_param_1.in/bifurcation_diagram_2026-03-13.png')
plt.show()

In [ ]:
eigs = []
for monodromy in Monodromy:
    eigenvalues, _ = np.linalg.eig(monodromy)
    sorted_eigs = np.sort(np.abs(eigenvalues),stable=True)
    eigs.append(sorted_eigs)

print(len(eigs), len(I_vals))

max1 = [eigs[i][-1] for i in range(len(eigs))]
max2 = [eigs[i][-2] for i in range(len(eigs))]
max3 = [eigs[i][-3] for i in range(len(eigs))]

In [ ]:
plot_eigenvalues(Monodromy[0],I_vals[0], T_sols[0])

In [ ]:
fig1, ax1 = plt.subplots(figsize=(18/2.54, 14/2.54), dpi=300)
#Set the font size for the axes labels and title
plt.rcParams.update({'font.size': 18})
ind = len(I_vals)
# Plot the eigenvalues
ax1.plot(I_vals[:ind], max1,marker='+', linestyle='-', color='blue', label=r'$\mu_1$')
# ax1.plot(I_vals[ind-1:], max1[ind-1:],'--', color='red', label='I >= 1.0')
ax1.plot(I_vals[:ind], max2, linestyle='--', color='red', label=r'$\mu_2$')
ax1.plot(I_vals[:ind], max3, linestyle='-.', color='green', label=r'$\mu_3$')
ax1.set_xlabel(f'$I$', fontsize=14)
ax1.set_ylabel(r'$|\mu|$', fontsize=14)
# ax1.set_xticks(np.arange(1, 1.3, 0.026))
ax1.tick_params(axis='both', labelsize=14)

# ax1.set_title(r'The $2$ most dominant Floquet Multipliers of the Monodromy Matrix vs Intensity I')
# ax1.set_yticks(np.arange(0.9, 1.001, 0.01))
# ax1.grid(which='both')
ax1.legend(loc='upper center', fontsize=8)
plt.tight_layout()


In [ ]:
orbit_finder = orbit(model.dydt,y0,model.T_ini, model.jacobian, solve_ivp, model.method,100,model.max_iter, model.precision)
k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged, mass, monodromy = run(model,f,J,model.n_z,"Newton_mass_conserv4",
                                                                         model.p0,model.T_ini,y0, filename=None, t_scale=True)

In [ ]:
model.I = 0.5*Ic


In [ ]:
#Using fixed point iteration to find the stationary solution
y_fp = (1/(2*np.pi))*np.ones_like(z_centers)+ 0.1*np.sin(z_centers)
# y_fp = y0# sol.y[:, -1] #use the last time point as an initial guess for the fixed point iteration
print(sp.integrate.trapezoid(y_fp, z_centers))
from scipy.optimize import fixed_point
def fixed_point_func(y):
    normalizer = sp.integrate.trapezoid(np.exp(-model.V_kuramoto(z_centers,y)), z_centers)
    return np.exp(-model.V_kuramoto(z_centers,y))/normalizer

y_fp = fixed_point(fixed_point_func, y_fp,method='del2', xtol=1e-8, maxiter=9000)
plt.plot(z_centers, y_fp, color='red', linewidth=1.5, label=r'Fixed Point Solution')
plt.xlabel('z',fontsize =18)
plt.ylabel(r'$\rho(z)$', fontsize = 18)
plt.title('Fixed Point Solution of the Density')
plt.legend()
plt.grid()
plt.show()


In [ ]:
model.n_z = 100
model.update_params()
rho =  np.random.rand(model.n_z-1)
model.I = 1.5
Jac = model.jacobian(0, rho)
# Jac_I = model.df_dI(0, rho)
f = model.dydt

def compute_jacobian(f, x, epsi=1e-5):
    """    Compute the Jacobian of a vector function f at point x using finite differences.
    """

    #Do not use explilicite loop to compute the Jacobian
    n = len(x)
    m = len(f(0,x))
    J = np.zeros((m, n))

    for i in range(m):
        for j in range(n):
            x_plus = np.copy(x)
            x_minus = np.copy(x)
            x_plus[j] += epsi
            x_minus[j] -= epsi
            J[i, j] = (f(0,x_plus)[i] - f(0,x_minus)[i]) / (2 * epsi)
      
    return J

J_num = compute_jacobian(f,rho, epsi=1e-5)
# J_num_I = df_dI(f, rho, model.I, epsi=1e-5)
#Compare the two Jacobians
print('Difference between analytical and numerical Jacobian:', np.linalg.norm(Jac - J_num))
#Check that the Jacobian is correct using the definition

# model.jacobian(0, rho) @ (rho) - f(0, rho)
# print(Jac @ (rho) - f(0, rho))

In [ ]:
#Checking the jacobian function
R = []
H = [1e-1,1e-2,1e-3,1e-4,1e-5,1e-6,1e-7,1e-8]
for eps in H:
    r = f(0, rho + eps*rho) - f(0, rho)  - Jac @ (eps*rho)
    R.append(np.linalg.norm(r))
print("Residuals for different epsilons:", R)

plt.figure(figsize=(10, 6))
plt.plot(H, R, marker='o')
#plot the slope h***2
plt.plot(H, [R[0] * (h / H[0])**2 for h in H], marker='*',linestyle='--', color='red', label='Slope ~ h^2')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Epsilon (h)')
plt.ylabel('Residual Norm')
plt.legend()
plt.title('Residual Norm vs Epsilon')
plt.grid(True)